# Phase 01 — DINOv3 model smoke check

First verify that DINOv3 loads and produces an image embedding on the Colab GPU. This does not need CIFAR-10 yet. Use a real CIFAR image for the first dataset check.

In [ ]:
!pip -q install -U transformers accelerate pillow

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

If the model repository asks for authentication, log in to Hugging Face using a read token. Do not paste the token into a notebook cell or commit it to GitHub.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
model_name = 'facebook/dinov3-vits16-pretrain-lvd1689m'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device).eval()
print('Loaded:', model_name)
print('Device:', device)

In [ ]:
from PIL import Image
import torch.nn.functional as F

# Temporary image proves the model runs; CIFAR-10 will replace this next.
image = Image.new('RGB', (224, 224), color=(120, 160, 200))
inputs = processor(images=image, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.inference_mode():
    output = model(**inputs)
    embedding = F.normalize(output.last_hidden_state[:, 0].float(), p=2, dim=-1)
print('Token output:', tuple(output.last_hidden_state.shape))
print('Embedding:', tuple(embedding.shape))
print('Embedding finite:', torch.isfinite(embedding).all().item())
print('Embedding norm:', embedding.norm(dim=-1).tolist())

## CIFAR-10 first run

This quick run samples 50 images per class to validate the full pipeline without extracting all 50,000 training images. Classes 0–7 are treated as known; classes 8–9 are held out as unknowns. This is a smoke experiment, not the Phase 02 final benchmark.

In [ ]:
!pip -q install datasets scikit-learn hdbscan

In [ ]:
from datasets import load_dataset
import numpy as np

dataset = load_dataset('cifar10')
train = dataset['train']
class_names = train.features['label'].names
labels_all = np.asarray(train['label'])

# Balanced small sample: 50 images from each CIFAR-10 class.
rng = np.random.default_rng(42)
chosen = np.concatenate([rng.choice(np.flatnonzero(labels_all == c), 50, replace=False) for c in range(10)])
images = [train[int(i)]['img'].convert('RGB') for i in chosen]
labels = labels_all[chosen]
print('Sample images:', len(images), '| class counts:', np.bincount(labels))
print('Classes:', class_names)

In [ ]:
def encode_images(images, batch_size=16):
    vectors = []
    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size]
        inputs = processor(images=batch, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = model(**inputs).last_hidden_state[:, 0].float()
            out = F.normalize(out, p=2, dim=-1)
        vectors.append(out.cpu().numpy())
    return np.concatenate(vectors).astype('float32')

embeddings = encode_images(images)
print('Embedding array:', embeddings.shape, embeddings.dtype)
print('All finite:', np.isfinite(embeddings).all())

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

UNKNOWN = -1
known_classes = np.arange(8)
known_mask = np.isin(labels, known_classes)
train_idx, val_idx, test_idx, unknown_idx = [], [], [], []
for c in known_classes:
    ids = rng.permutation(np.flatnonzero(labels == c))
    train_idx.extend(ids[:30]); val_idx.extend(ids[30:40]); test_idx.extend(ids[40:])
unknown_idx = np.flatnonzero(np.isin(labels, [8, 9])).tolist()

# One normalized mean prototype per known class.
prototypes = np.stack([embeddings[np.asarray(train_idx)][labels[train_idx] == c].mean(axis=0) for c in known_classes])
prototypes /= np.linalg.norm(prototypes, axis=1, keepdims=True).clip(min=1e-12)
def nearest_distances(x):
    x = x / np.linalg.norm(x, axis=1, keepdims=True).clip(min=1e-12)
    return np.clip(1.0 - x @ prototypes.T, 0, 2)

# Calibrate the reject threshold on known validation samples only.
val_dist = nearest_distances(embeddings[val_idx]).min(axis=1)
threshold = float(np.quantile(val_dist, 0.95))
eval_idx = np.asarray(test_idx + unknown_idx)
d = nearest_distances(embeddings[eval_idx])
scores = d.min(axis=1)
pred = known_classes[d.argmin(axis=1)].astype(int)
pred[scores > threshold] = UNKNOWN
truth = labels[eval_idx].copy().astype(int)
truth[np.isin(truth, [8, 9])] = UNKNOWN
known_eval = truth != UNKNOWN
unknown_eval = ~known_eval
metrics = {
    'known_accuracy': float(accuracy_score(truth[known_eval], pred[known_eval])),
    'unknown_f1': float(f1_score(unknown_eval, pred == UNKNOWN, zero_division=0)),
    'unknown_auroc': float(roc_auc_score(unknown_eval.astype(int), scores)),
    'threshold': threshold,
}
print(metrics)

In [ ]:
import hdbscan
from sklearn.metrics import adjusted_rand_score

clusterer = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5)
cluster_ids = clusterer.fit_predict(embeddings)
print('Clusters (noise is -1):', np.unique(cluster_ids, return_counts=True))
print('Exploratory ARI vs class labels:', round(adjusted_rand_score(labels, cluster_ids), 4))

## Save run artifacts to Google Drive

Run this cell after the pipeline completes. It saves the sampled embeddings, labels, prototypes, threshold, and run configuration. The dataset and pretrained weights are not copied into the project folder.

In [ ]:
from google.colab import drive
from pathlib import Path
import json

drive.mount('/content/drive')
out_dir = Path('/content/drive/MyDrive/open-vision-memory/phase01')
out_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(out_dir / 'cifar10_sample_embeddings.npz', embeddings=embeddings, labels=labels, cluster_ids=cluster_ids)
np.savez_compressed(out_dir / 'prototype_memory.npz', labels=known_classes, prototypes=prototypes, threshold=np.array(threshold))
config = {'backbone': model_name, 'pooling': 'CLS token', 'normalization': 'L2', 'sample_per_class': 50, 'known_classes': [int(x) for x in known_classes], 'unknown_classes': [8, 9], 'threshold_quantile': 0.95, 'threshold': threshold, 'metrics': metrics}
(out_dir / 'run_config.json').write_text(json.dumps(config, indent=2))
print('Saved artifacts to:', out_dir)